In [ ]:
# Databricks notebook source
import os
import sys
from datetime import datetime, timezone

try:
    script_dir = os.path.dirname(__file__)
except NameError:
    script_dir = os.getcwd()

for p in [script_dir, os.path.abspath(os.path.join(script_dir, "../..")), os.path.abspath(os.path.join(script_dir, ".."))]:
    if p not in sys.path:
        sys.path.insert(0, p)

from pyspark.sql.functions import col, from_json, current_timestamp, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from src.common.logger import CentralLogger


def stream_topic_to_delta(
    spark,
    logger,
    topic_name: str,
    target_table: str,
    task_name: str,
    json_schema: StructType,
    checkpoint_path: str,
    kafka_server: str,
    jaas_config: str
):
    """
    Generic runner for consuming a Confluent Kafka topic and persisting to a Bronze Delta table.
    """
    start_time = datetime.now(timezone.utc)
    pipeline_name = "bronze_kafka_ingestion"

    try:
        # 1. Read Stream from Kafka topic with failOnDataLoss set to false for offset/partition safety
        raw_kafka_stream = (
            spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", kafka_server)
            .option("subscribe", topic_name)
            .option("kafka.security.protocol", "SASL_SSL")
            .option("kafka.sasl.mechanism", "PLAIN")
            .option("kafka.sasl.jaas.config", jaas_config)
            .option("kafka.ssl.endpoint.identification.algorithm", "https")
            .option("startingOffsets", "earliest")
            .option("failOnDataLoss", "false")
            .load()
        )

        # 2. Parse Binary Payload & Extract JSON Schema Fields
        parsed_df = (
            raw_kafka_stream
            .select(
                col("key").cast("string").alias("record_key"),
                from_json(col("value").cast("string"), json_schema).alias("data"),
                col("timestamp").alias("kafka_timestamp")
            )
            .select("record_key", "data.*", "kafka_timestamp")
            .withColumn("_ingestion_timestamp", current_timestamp())
            .withColumn("_source", expr(f"'confluent_kafka_{topic_name}'"))
        )

        # 3. Write Stream into Unity Catalog Bronze Delta Table
        streaming_query = (
            parsed_df.writeStream
            .format("delta")
            .outputMode("append")
            .trigger(availableNow=True)
            .option("checkpointLocation", checkpoint_path)
            .toTable(target_table)
        )

        streaming_query.awaitTermination()

        records_processed = sum(progress.get("numInputRows", 0) for progress in streaming_query.recentProgress)

        # 4. Log Success to Audit Table
        logger.log_execution(
            pipeline_name=pipeline_name,
            task_name=task_name,
            execution_status="SUCCESS",
            start_time=start_time,
            records_processed=records_processed,
            records_failed=0
        )
        print(f"Successfully processed {records_processed} records from topic '{topic_name}' into '{target_table}'.")

    except Exception as e:
        logger.log_execution(
            pipeline_name=pipeline_name,
            task_name=task_name,
            execution_status="FAILED",
            start_time=start_time,
            records_processed=0,
            records_failed=0,
            error=e
        )
        print(f"Error processing Kafka topic '{topic_name}': {str(e)}")
        raise e


def run_kafka_ingestion():
    logger = CentralLogger(spark)
    kafka_server = "pkc-w77k7w.centralus.azure.confluent.cloud:9092"

    # Credentials from Key Vault Secret Scope
    kafka_key = dbutils.secrets.get(scope="maven-secrets", key="kafka-api-key")
    kafka_secret = dbutils.secrets.get(scope="maven-secrets", key="kafka-api-secret")
    jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{kafka_key}" password="{kafka_secret}";'

    # --- TOPIC 1: ORDER STREAM ---
    orders_schema = StructType([
        StructField("transaction_date", StringType(), True),
        StructField("stock_date", StringType(), True),
        StructField("product_id", IntegerType(), True),
        StructField("customer_id", IntegerType(), True),
        StructField("store_id", IntegerType(), True),
        StructField("quantity", IntegerType(), True)
    ])

    stream_topic_to_delta(
        spark=spark,
        logger=logger,
        topic_name="order-stream",
        target_table="dbx_maven_market.bronze.raw_orders",
        task_name="raw_orders",
        json_schema=orders_schema,
        checkpoint_path="/Volumes/dbx_maven_market/default/my_volume/checkpoints/order_stream_v1/",
        kafka_server=kafka_server,
        jaas_config=jaas_config
    )

    # --- TOPIC 2: INVENTORY STREAM ---
    inventory_schema = StructType([
        StructField("inventory_date", StringType(), True),
        StructField("store_id", IntegerType(), True),
        StructField("product_id", IntegerType(), True),
        StructField("stock_on_hand", IntegerType(), True),
        StructField("reorder_flag", IntegerType(), True)
    ])

    stream_topic_to_delta(
        spark=spark,
        logger=logger,
        topic_name="inventory-stream",
        target_table="dbx_maven_market.bronze.raw_inventory",
        task_name="raw_inventory",
        json_schema=inventory_schema,
        checkpoint_path="/Volumes/dbx_maven_market/default/my_volume/checkpoints/inventory_stream_v1/",
        kafka_server=kafka_server,
        jaas_config=jaas_config
    )


# Execute streaming pipeline
run_kafka_ingestion()

In [0]:
%sql
-- COMMAND --------------
SELECT 'raw_orders' AS stream_table, COUNT(*) AS row_count FROM dbx_maven_market.bronze.raw_orders
UNION ALL
SELECT 'raw_inventory' AS stream_table, COUNT(*) AS row_count FROM dbx_maven_market.bronze.raw_inventory;

total_records
10568
